# 组合优化策略

本文档展示如何将组合优化与策略回测框架结合，构建完整的量化投资策略。

> **前置阅读**：
> - 组合优化模块的架构和目标/约束条件 API，请参阅 **[基本框架](基本框架.ipynb)**
> - 均值方差模型的使用，请参阅 **[均值方差模型](均值方差模型.ipynb)**
> - 风险预算/风险平价模型的使用，请参阅 **[风险预算模型](风险预算模型.ipynb)**
> - 策略回测框架的使用，请参阅 **[策略回测](../回测框架/策略回测.ipynb)**
> - 回测框架架构，请参阅 **[基本框架](../回测框架/基本框架.ipynb)**

## 架构概述

将组合优化嵌入策略回测的核心思路是：通过继承 `MakeStrategy` 并重写 `genSignal` 方法，在每个再平衡时点调用 CVXPC 求解最新的最优权重作为交易信号。

```
每个再平衡时点:
  ├── 获取当前因子数据（预期收益、协方差阵等）
  ├── 构造 OptimizationObjective + Constraints
  ├── CVXPC.solve() → 最优权重
  └── 返回权重作为交易信号 → MakeAccount 撮合成交
```

策略回测的完整流程（因子计算 → 信号生成 → 撮合成交 → 报告生成）请参阅 **[策略回测](../回测框架/策略回测.ipynb)**。

In [ ]:
# 全局设置
import os
import datetime as dt
import warnings
warnings.filterwarnings(action='ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['SimHei']
matplotlib.rcParams['axes.unicode_minus'] = False
from IPython.display import HTML

from QuantStudio.Factor.HDF5DB import HDF5DB
from QuantStudio.Risk.HDF5RDB import HDF5FRDB
from QuantStudio.Tools.DateTimeFun import getMonthLastDateTime

In [ ]:
# 参数设置
HDB = HDF5DB(args={"MainDir": "../data/HDF5"}).connect()
RDB = HDF5FRDB(args={"MainDir": "../data/Risk"}).connect()

StartDT, EndDT = dt.datetime(2025, 1, 1), dt.datetime(2025, 3, 31)
TestStartDT, TestEndDT = dt.datetime(2025, 1, 31), EndDT

FT = HDB.getTable("stock_cn_day_bar")
DTRuler = FT.getDateTime(start_dt=StartDT, end_dt=EndDT)
TestDTs = FT.getDateTime(start_dt=TestStartDT, end_dt=TestEndDT)
SectionIDs = IDs = FT.getID()

BalanceDTs = getMonthLastDateTime(DTRuler)  # 月末再平衡

## 策略一：最小方差策略

每期月末求解最小方差组合（$\gamma=0, \lambda=1$），约束条件为纯多头 + 全额投资。

核心逻辑在 `genSignal` 方法中：
1. 从 `x` 中获取当前时点的股票池 mask 和协方差矩阵
2. 构造 `MeanVarianceObjective`（ExpectedReturnCoef=0）
3. 添加基本约束条件
4. 调用 `CVXPC.solve()` 求解最优权重
5. 返回权重作为目标权重信号

In [ ]:
from QuantStudio.Core.CalcEngine import Engine
from QuantStudio.Core.Node import DTLocalContext, DTInitData
import QuantStudio.Factor.FactorOperator as fo
from QuantStudio.Factor.FactorCache import FeatherFactorCache
from QuantStudio.Factor.Factor import FactorContext
from QuantStudio.BackTest.BackTestModel import BTReport
from QuantStudio.BackTest.Strategy.Strategy import MakeStrategy, AccountReport
from QuantStudio.PortfolioConstructor.BasePC import MeanVarianceObjective, WeightConstraint, BudgetConstraint
from QuantStudio.PortfolioConstructor.CVXPC import CVXPC

class MakeMinVarStrategy(MakeStrategy):
    """最小方差组合策略"""
    def genSignal(self, f, idt, x, last_price, cash, position_num, args):
        Mask, Cov = x[0].loc[idt].values, x[1].loc[idt].values
        IDs = x[0].columns.to_list()

        Objective = MeanVarianceObjective(
            mask=Mask, cov=Cov,
            args={"ExpectedReturnCoef": 0, "RiskAversionCoef": 1}
        )
        ConstraintList = [
            WeightConstraint(mask=Mask, up_limit=1, down_limit=0),
            BudgetConstraint(mask=Mask, args={"UpLimit": 1, "DownLimit": 1})
        ]

        PC = CVXPC(mask=Mask, objective=Objective, constraints=ConstraintList, args={})
        Portfolio, Info = PC.solve()
        return pd.Series(np.where(np.abs(Portfolio) > 1e-8, Portfolio, 0), index=IDs).fillna(0)

In [ ]:
FT = HDB.getTable("stock_cn_day_bar")
Price = FT.getFactor("close")

FT = HDB.getTable("stock_cn_status")
IfListed = FT.getFactor("if_listed")
Mask = (fo.NotNull()(Price) & (IfListed == 1))

RT = RDB.getTable("demo_risk_table")

makeMinVarStrategy = MakeMinVarStrategy(
    signal_type="目标权重", init_cash=1e6, start_dt=TestDTs[0],
    x_lookback=[0], x_section_ids=[None], signal_dts=BalanceDTs
)
Strategy = makeMinVarStrategy(
    Mask, last_price=Price,
    extra_deps=[RT], extra_section_ids=[SectionIDs], extra_lookback=[0]
)

FT = HDB.getTable("index_cn_day_bar")
BmkNV = FT.getFactor("close", args={"SectionIDs": ["000905.SH"]})

StrategyReport = AccountReport(account=Strategy, bmk_nv=BmkNV, args={"GenReport": True})
NodeList = [StrategyReport]
Report = BTReport(result_nodes=NodeList)

with FeatherFactorCache(args={"DTRuler": DTRuler, "CacheDir": "../data/Cache", "StartMode": "new"}) as Cache:
    with FactorContext(DTRuler=DTRuler, SectionIDs=SectionIDs, DataCache=Cache) as Context:
        with Engine() as ExecEngine:
            Rslt = ExecEngine.run([Report], Context,
                                 fwd_data_list=[DTLocalContext(DTs=TestDTs)],
                                 init_data_list=[DTInitData(DTRange=(TestDTs[0], TestDTs[-1]))])

display(HTML(Rslt[0]["Report"]))

## 策略二：风险平价策略

每期月末求解风险平价组合（等风险预算），约束条件为纯多头 + 全额投资。

> **求解器选择**：风险预算问题涉及指数锥约束，需要使用 CLARABEL 等支持指数锥的求解器。

In [ ]:
import cvxpy as cvx
from QuantStudio.PortfolioConstructor.BasePC import RiskBudgetObjective

class MakeRiskParityStrategy(MakeStrategy):
    """风险平价组合策略"""
    def genSignal(self, f, idt, x, last_price, cash, position_num, args):
        Mask, Cov = x[0].loc[idt].values, x[1].loc[idt].values
        IDs = x[0].columns.to_list()

        Objective = RiskBudgetObjective(mask=Mask, cov=Cov)
        ConstraintList = [
            WeightConstraint(mask=Mask, up_limit=1, down_limit=0),
            BudgetConstraint(mask=Mask, args={"UpLimit": 1, "DownLimit": 1})
        ]

        PC = CVXPC(mask=Mask, objective=Objective, constraints=ConstraintList,
                   args={"OptimOption": {"solver": cvx.CLARABEL}})
        Portfolio, Info = PC.solve()
        return pd.Series(np.where(np.abs(Portfolio) > 1e-8, Portfolio, 0), index=IDs).fillna(0)

In [ ]:
makeRiskParityStrategy = MakeRiskParityStrategy(
    signal_type="目标权重", init_cash=1e6, start_dt=TestDTs[0],
    x_lookback=[0], x_section_ids=[None], signal_dts=BalanceDTs
)
Strategy = makeRiskParityStrategy(
    Mask, last_price=Price,
    extra_deps=[RT], extra_section_ids=[SectionIDs], extra_lookback=[0]
)

StrategyReport = AccountReport(account=Strategy, bmk_nv=BmkNV, args={"GenReport": True})
NodeList = [StrategyReport]
Report = BTReport(result_nodes=NodeList)

with FeatherFactorCache(args={"DTRuler": DTRuler, "CacheDir": "../data/Cache", "StartMode": "new"}) as Cache:
    with FactorContext(DTRuler=DTRuler, SectionIDs=SectionIDs, DataCache=Cache) as Context:
        with Engine() as ExecEngine:
            Rslt = ExecEngine.run([Report], Context,
                                 fwd_data_list=[DTLocalContext(DTs=TestDTs)],
                                 init_data_list=[DTInitData(DTRange=(TestDTs[0], TestDTs[-1]))])

display(HTML(Rslt[0]["Report"]))

## 策略三：带约束的均值方差策略

在实际策略中，通常需要添加多种约束条件来控制风险暴露和交易成本。下面的示例展示了一个更完善的均值方差策略：

- 个股权重上限 10%
- 组合年化波动率不超过 15%
- 总换手率不超过 50%
- 波动率约束设置了 `DropPriority=1`，当问题不可行时优先松弛

In [ ]:
from QuantStudio.PortfolioConstructor.BasePC import VolatilityConstraint, TurnoverConstraint

class MakeConstrainedMVStrategy(MakeStrategy):
    """带约束的均值方差策略"""
    def genSignal(self, f, idt, x, last_price, cash, position_num, args):
        Mask, Cov, ExpectedReturn, P0 = (
            x[0].loc[idt].values, x[1].loc[idt].values,
            x[2].loc[idt].values, x[3].loc[idt].values
        )
        IDs = x[0].columns.to_list()

        Objective = MeanVarianceObjective(
            mask=Mask, expected_return=ExpectedReturn, p0=P0, cov=Cov,
            args={"ExpectedReturnCoef": 0.5, "RiskAversionCoef": 2}
        )
        ConstraintList = [
            WeightConstraint(mask=Mask, up_limit=1, down_limit=0),
            WeightConstraint(mask=Mask, up_limit=0.10),
            BudgetConstraint(mask=Mask, args={"UpLimit": 1, "DownLimit": 1}),
            VolatilityConstraint(mask=Mask, cov=Cov,
                                 args={"UpLimit": 0.15, "DropPriority": 1}),
            TurnoverConstraint(mask=Mask, p0=P0,
                               args={"ConstraintType": "总换手限制", "UpLimit": 0.5}),
        ]

        PC = CVXPC(mask=Mask, objective=Objective, constraints=ConstraintList, args={})
        Portfolio, Info = PC.solve()
        return pd.Series(np.where(np.abs(Portfolio) > 1e-8, Portfolio, 0), index=IDs).fillna(0)

In [ ]:
# 获取预期收益因子（示例：使用 EP TTM 作为预期收益的代理变量）
FT = HDB.getTable("stock_cn_factor_value")
EP = FT.getFactor("ep_ttm")

# 上期持仓权重（从 Account 因子获取，这里用等权近似）
nStock = (~np.isnan(Price.iloc[0].values)).sum()

makeConstrainedMVStrategy = MakeConstrainedMVStrategy(
    signal_type="目标权重", init_cash=1e6, start_dt=TestDTs[0],
    x_lookback=[0], x_section_ids=[None], signal_dts=BalanceDTs
)
Strategy = makeConstrainedMVStrategy(
    Mask, last_price=Price,
    extra_deps=[RT, EP, Mask],  # Mask 用作 p0 占位（实际使用时替换为上期持仓）
    extra_section_ids=[SectionIDs, SectionIDs, SectionIDs],
    extra_lookback=[0, 0, 0]
)

StrategyReport = AccountReport(account=Strategy, bmk_nv=BmkNV, args={"GenReport": True})
NodeList = [StrategyReport]
Report = BTReport(result_nodes=NodeList)

with FeatherFactorCache(args={"DTRuler": DTRuler, "CacheDir": "../data/Cache", "StartMode": "new"}) as Cache:
    with FactorContext(DTRuler=DTRuler, SectionIDs=SectionIDs, DataCache=Cache) as Context:
        with Engine() as ExecEngine:
            Rslt = ExecEngine.run([Report], Context,
                                 fwd_data_list=[DTLocalContext(DTs=TestDTs)],
                                 init_data_list=[DTInitData(DTRange=(TestDTs[0], TestDTs[-1]))])

display(HTML(Rslt[0]["Report"]))

## 策略四：最大分散化策略

每期月末求解最大分散化组合，约束条件为纯多头 + 全额投资。

最大分散化组合不需要预期收益作为输入，仅依赖协方差矩阵，对参数估计误差的敏感度低于均值方差模型。

In [ ]:
from QuantStudio.PortfolioConstructor.BasePC import MaxDiversificationObjective

class MakeMaxDivStrategy(MakeStrategy):
    """最大分散化组合策略"""
    def genSignal(self, f, idt, x, last_price, cash, position_num, args):
        Mask, Cov = x[0].loc[idt].values, x[1].loc[idt].values
        IDs = x[0].columns.to_list()

        Objective = MaxDiversificationObjective(mask=Mask, cov=Cov)
        ConstraintList = [
            WeightConstraint(mask=Mask, up_limit=1, down_limit=0),
            BudgetConstraint(mask=Mask, args={"UpLimit": 1, "DownLimit": 1})
        ]

        PC = CVXPC(mask=Mask, objective=Objective, constraints=ConstraintList, args={})
        Portfolio, Info = PC.solve()
        return pd.Series(np.where(np.abs(Portfolio) > 1e-8, Portfolio, 0), index=IDs).fillna(0)

In [ ]:
makeMaxDivStrategy = MakeMaxDivStrategy(
    signal_type="目标权重", init_cash=1e6, start_dt=TestDTs[0],
    x_lookback=[0], x_section_ids=[None], signal_dts=BalanceDTs
)
Strategy = makeMaxDivStrategy(
    Mask, last_price=Price,
    extra_deps=[RT], extra_section_ids=[SectionIDs], extra_lookback=[0]
)

StrategyReport = AccountReport(account=Strategy, bmk_nv=BmkNV, args={"GenReport": True})
NodeList = [StrategyReport]
Report = BTReport(result_nodes=NodeList)

with FeatherFactorCache(args={"DTRuler": DTRuler, "CacheDir": "../data/Cache", "StartMode": "new"}) as Cache:
    with FactorContext(DTRuler=DTRuler, SectionIDs=SectionIDs, DataCache=Cache) as Context:
        with Engine() as ExecEngine:
            Rslt = ExecEngine.run([Report], Context,
                                 fwd_data_list=[DTLocalContext(DTs=TestDTs)],
                                 init_data_list=[DTInitData(DTRange=(TestDTs[0], TestDTs[-1]))])

display(HTML(Rslt[0]["Report"]))

## genSignal 方法详解

`genSignal` 是在每个再平衡时点被回调的核心方法，其签名和参数含义如下：

```python
def genSignal(
    self,
    f: PanelOperation,           # 策略因子对象（算子自身）
    idt: dt.datetime,            # 当前时点
    x: List[pd.DataFrame],       # 策略依赖的因子数据
    last_price: pd.Series,       # 当前证券最新价
    cash: float,                 # 当前账户剩余现金
    position_num: pd.Series,     # 当前账户持仓数量
    args: dict                   # 额外参数字典
) -> None | pd.Series:
```

### 通过 x 传递数据

`x` 是一个列表，每个元素是对应时点的 DataFrame（index=[datetime], columns=[证券ID]）。数据来源由 `__call__` 中的参数决定：

| 数据来源 | 参数 | 说明 |
|----------|------|------|
| 直接传入的因子 | `*x` 位置参数 | 通过 `x_lookback` 和 `x_section_ids` 控制回溯和截面 |
| 额外依赖节点 | `extra_deps` | 通过 `extra_section_ids` 和 `extra_lookback` 控制 |

`x` 列表的顺序为：先直接传入的因子（按 `__call__` 中 `*x` 的顺序），后额外依赖节点（按 `extra_deps` 的顺序）。

### 典型模式

```python
class MyStrategy(MakeStrategy):
    def genSignal(self, f, idt, x, last_price, cash, position_num, args):
        # x[0], x[1], ... 对应 __call__ 中传入的因子（或 extra_deps）
        Mask = x[0].loc[idt].values        # 当前时点的股票池
        Cov = x[1].loc[idt].values          # 当前时点的协方差阵
        ExpectedReturn = x[2].loc[idt].values  # 当前时点的预期收益

        # 构造优化问题
        Objective = MeanVarianceObjective(mask=Mask, expected_return=ExpectedReturn, cov=Cov, args={...})
        Constraints = [...]

        # 求解
        PC = CVXPC(mask=Mask, objective=Objective, constraints=Constraints)
        Portfolio, Info = PC.solve()

        # 返回目标权重信号
        return pd.Series(Portfolio, index=x[0].columns)
```

### 注意事项

1. **求解失败处理**：当 `CVXPC.solve()` 返回 `Portfolio=None` 时，`genSignal` 返回 None 表示当期不调仓
2. **约束松弛**：通过设置约束的 `DropPriority` 控制松弛优先级，CVXPC 在求解失败时会自动尝试松弛
3. **求解器选择**：
   - 均值方差/最大分散化（纯二次规划）：OSQP、SCIP 均可
   - 风险预算（指数锥）：必须使用 CLARABEL
   - 含整数变量（NonZeroNum 约束）：必须使用 SCIP
4. **性能考虑**：每期都需要求解优化问题，对于大规模股票池（如全市场 5000+ 只股票），建议先用因子筛选缩小候选池

## 多策略对比

可以在同一报告中对比多个策略的表现。下面展示如何同时运行最小方差和风险平价策略。

In [ ]:
# 同时运行两个策略
makeMinVarStrategy = MakeMinVarStrategy(
    signal_type="目标权重", init_cash=1e6, start_dt=TestDTs[0],
    x_lookback=[0], x_section_ids=[None], signal_dts=BalanceDTs
)
MinVarStrategy = makeMinVarStrategy(
    Mask, last_price=Price,
    extra_deps=[RT], extra_section_ids=[SectionIDs], extra_lookback=[0]
)

makeRiskParityStrategy = MakeRiskParityStrategy(
    signal_type="目标权重", init_cash=1e6, start_dt=TestDTs[0],
    x_lookback=[0], x_section_ids=[None], signal_dts=BalanceDTs
)
RiskParityStrategy = makeRiskParityStrategy(
    Mask, last_price=Price,
    extra_deps=[RT], extra_section_ids=[SectionIDs], extra_lookback=[0]
)

MinVarReport = AccountReport(account=MinVarStrategy, bmk_nv=BmkNV, args={"GenReport": True, "Name": "最小方差"})
RiskParityReport = AccountReport(account=RiskParityStrategy, bmk_nv=BmkNV, args={"GenReport": True, "Name": "风险平价"})

NodeList = [MinVarReport, RiskParityReport]
Report = BTReport(result_nodes=NodeList)

with FeatherFactorCache(args={"DTRuler": DTRuler, "CacheDir": "../data/Cache", "StartMode": "new"}) as Cache:
    with FactorContext(DTRuler=DTRuler, SectionIDs=SectionIDs, DataCache=Cache) as Context:
        with Engine() as ExecEngine:
            Rslt = ExecEngine.run([Report], Context,
                                 fwd_data_list=[DTLocalContext(DTs=TestDTs)],
                                 init_data_list=[DTInitData(DTRange=(TestDTs[0], TestDTs[-1]))])

# 展示两个报告
for key in Rslt[0]:
    if "Report" in Rslt[0][key]:
        display(HTML(f"<h3>{key}</h3>"))
        display(HTML(Rslt[0][key]["Report"]))

## 最佳实践

### 模型选择指南

| 场景 | 推荐模型 | 原因 |
|------|----------|------|
| 有可靠预期收益预测 | 均值方差 | 充分利用收益信息 |
| 无预期收益预测，追求稳健 | 风险平价 | 不依赖收益估计，分散化好 |
| 无预期收益预测，追求分散化 | 最大分散化 | 最大化风险分散收益 |
| 指数增强 | 均值方差（相对基准） | 控制跟踪误差，追求超额收益 |
| 绝对收益/纯风险控制 | 最小方差 | 仅控制风险，不依赖收益预测 |

### 约束条件建议

1. **始终添加权重约束**：至少设置 `up_limit=1, down_limit=0`（纯多头）
2. **始终添加预算约束**：通常设置 `UpLimit=1, DownLimit=1`（全额投资）
3. **适度设置个股权重上限**：推荐 5%~15%，避免过度集中
4. **换手控制**：通过 `TurnoverConstraint` 或目标函数中的换手惩罚系数控制
5. **波动率约束设 DropPriority**：波动率约束是"软约束"，设置松弛优先级避免问题不可行

### 协方差矩阵估计

协方差矩阵是组合优化的核心输入，建议：
- 优先使用多因子风险模型（Barra 等）估计的协方差阵，而非样本协方差
- 使用时通过 `factor_cov` + `factor_data` + `specific_risk` 参数传入
- 如果使用样本协方差，考虑使用压缩估计（shrinkage）改善估计质量